# 检测 GRIB 数据时间

目的：
1. 检查 uv100.grib 的时间范围
2. 确认时间是 UTC 还是北京时间 (UTC+8)
3. 分析训练数据的时间覆盖情况

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

## 1. 加载 GRIB 数据

In [2]:
# 加载原始 grib 文件
uv100_path = "/home/xiao/Desktop/WFMPN/东北电网数据/uv100.grib"
uv100 = xr.open_dataset(uv100_path, engine="cfgrib")

print("=== uv100.grib 基本信息 ===")
print(f"变量: {list(uv100.data_vars)}")
print(f"坐标: {list(uv100.coords)}")
print(f"维度: {dict(uv100.dims)}")

=== uv100.grib 基本信息 ===
变量: ['u100', 'v100']
坐标: ['number', 'time', 'step', 'surface', 'latitude', 'longitude', 'valid_time']
维度: {'time': 51151, 'latitude': 65, 'longitude': 81}


## 2. 时间范围分析

In [3]:
# 获取时间坐标
times = pd.to_datetime(uv100.time.values)

print("=== 时间范围 ===")
print(f"起始时间: {times[0]}")
print(f"结束时间: {times[-1]}")
print(f"总时间点: {len(times)}")
print(f"时间跨度: {times[-1] - times[0]}")

=== 时间范围 ===
起始时间: 2020-01-01 00:00:00
结束时间: 2025-11-01 06:00:00
总时间点: 51151
时间跨度: 2131 days 06:00:00


In [4]:
# 检查时间间隔
time_diffs = np.diff(times)
unique_diffs = pd.Series(time_diffs).value_counts()

print("=== 时间间隔统计 ===")
print(unique_diffs)

=== 时间间隔统计 ===
0 days 01:00:00    51150
Name: count, dtype: int64


In [5]:
# 查看前 10 个时间点
print("=== 前 10 个时间点 ===")
for i, t in enumerate(times[:10]):
    print(f"  [{i}] {t}")

=== 前 10 个时间点 ===
  [0] 2020-01-01 00:00:00
  [1] 2020-01-01 01:00:00
  [2] 2020-01-01 02:00:00
  [3] 2020-01-01 03:00:00
  [4] 2020-01-01 04:00:00
  [5] 2020-01-01 05:00:00
  [6] 2020-01-01 06:00:00
  [7] 2020-01-01 07:00:00
  [8] 2020-01-01 08:00:00
  [9] 2020-01-01 09:00:00


In [6]:
# 查看后 10 个时间点
print("=== 后 10 个时间点 ===")
for i, t in enumerate(times[-10:]):
    print(f"  [{len(times)-10+i}] {t}")

=== 后 10 个时间点 ===
  [51141] 2025-10-31 21:00:00
  [51142] 2025-10-31 22:00:00
  [51143] 2025-10-31 23:00:00
  [51144] 2025-11-01 00:00:00
  [51145] 2025-11-01 01:00:00
  [51146] 2025-11-01 02:00:00
  [51147] 2025-11-01 03:00:00
  [51148] 2025-11-01 04:00:00
  [51149] 2025-11-01 05:00:00
  [51150] 2025-11-01 06:00:00


## 3. UTC vs 北京时间分析

ERA5 数据默认使用 **UTC 时间**（协调世界时）。

北京时间 = UTC + 8 小时

In [7]:
# 检查时间是否有时区信息
print("=== 时区信息 ===")
print(f"时间类型: {type(times[0])}")
print(f"时区信息: {times[0].tzinfo}")

# ERA5 数据说明
print("\n=== ERA5 时间说明 ===")
print("ERA5 再分析数据使用 UTC 时间（无时区标记）")
print("如需转换为北京时间，需要 +8 小时")
print(f"\n示例:")
print(f"  UTC:    {times[0]}")
print(f"  北京时间: {times[0] + timedelta(hours=8)}")

=== 时区信息 ===
时间类型: <class 'pandas._libs.tslibs.timestamps.Timestamp'>
时区信息: None

=== ERA5 时间说明 ===
ERA5 再分析数据使用 UTC 时间（无时区标记）
如需转换为北京时间，需要 +8 小时

示例:
  UTC:    2020-01-01 00:00:00
  北京时间: 2020-01-01 08:00:00


In [8]:
# 检查第一天的小时分布
first_day = times[0].date()
first_day_times = times[times.date == first_day]

print(f"=== 第一天 ({first_day}) 的时间点 ===")
for t in first_day_times:
    print(f"  {t.hour:02d}:00 UTC  ->  {(t.hour + 8) % 24:02d}:00 北京时间 (次日)" if t.hour >= 16 else f"  {t.hour:02d}:00 UTC  ->  {t.hour + 8:02d}:00 北京时间")

=== 第一天 (2020-01-01) 的时间点 ===
  00:00 UTC  ->  08:00 北京时间
  01:00 UTC  ->  09:00 北京时间
  02:00 UTC  ->  10:00 北京时间
  03:00 UTC  ->  11:00 北京时间
  04:00 UTC  ->  12:00 北京时间
  05:00 UTC  ->  13:00 北京时间
  06:00 UTC  ->  14:00 北京时间
  07:00 UTC  ->  15:00 北京时间
  08:00 UTC  ->  16:00 北京时间
  09:00 UTC  ->  17:00 北京时间
  10:00 UTC  ->  18:00 北京时间
  11:00 UTC  ->  19:00 北京时间
  12:00 UTC  ->  20:00 北京时间
  13:00 UTC  ->  21:00 北京时间
  14:00 UTC  ->  22:00 北京时间
  15:00 UTC  ->  23:00 北京时间
  16:00 UTC  ->  00:00 北京时间 (次日)
  17:00 UTC  ->  01:00 北京时间 (次日)
  18:00 UTC  ->  02:00 北京时间 (次日)
  19:00 UTC  ->  03:00 北京时间 (次日)
  20:00 UTC  ->  04:00 北京时间 (次日)
  21:00 UTC  ->  05:00 北京时间 (次日)
  22:00 UTC  ->  06:00 北京时间 (次日)
  23:00 UTC  ->  07:00 北京时间 (次日)


## 4. 训练数据时间覆盖分析

In [9]:
# 训练数据配置
TRAIN_SIZE = 43824  # 约5年
TEST_SIZE = 7324    # 剩余数据

print("=== 数据划分 ===")
print(f"总数据量: {len(times)} 小时")
print(f"训练集: {TRAIN_SIZE} 小时 ({TRAIN_SIZE/24:.0f} 天)")
print(f"测试集: {TEST_SIZE} 小时 ({TEST_SIZE/24:.0f} 天)")

if len(times) >= TRAIN_SIZE:
    train_end_time = times[TRAIN_SIZE - 1]
    print(f"\n训练集时间范围: {times[0]} ~ {train_end_time}")
    print(f"  (UTC 时间)")
    print(f"训练集北京时间: {times[0] + timedelta(hours=8)} ~ {train_end_time + timedelta(hours=8)}")

if len(times) >= TRAIN_SIZE + TEST_SIZE:
    test_start_time = times[TRAIN_SIZE]
    test_end_time = times[TRAIN_SIZE + TEST_SIZE - 1]
    print(f"\n测试集时间范围: {test_start_time} ~ {test_end_time}")
    print(f"  (UTC 时间)")

=== 数据划分 ===
总数据量: 51151 小时
训练集: 43824 小时 (1826 天)
测试集: 7324 小时 (305 天)

训练集时间范围: 2020-01-01 00:00:00 ~ 2024-12-30 23:00:00
  (UTC 时间)
训练集北京时间: 2020-01-01 08:00:00 ~ 2024-12-31 07:00:00

测试集时间范围: 2024-12-31 00:00:00 ~ 2025-11-01 03:00:00
  (UTC 时间)


## 5. 与精确点数据对齐分析

In [10]:
# 加载精确点数据日期
turbine_dates = np.load("data/turbine_points/turbine_dates.npy", allow_pickle=True)

print("=== 精确点数据日期范围 ===")
print(f"起始日期: {turbine_dates[0]}")
print(f"结束日期: {turbine_dates[-1]}")
print(f"总天数: {len(turbine_dates)}")

=== 精确点数据日期范围 ===
起始日期: 2024-09-23
结束日期: 2025-08-24
总天数: 266


In [11]:
# 分析重叠情况
grid_start = times[0].date()
grid_end = times[-1].date()

# 假设训练集结束于 TRAIN_SIZE
if len(times) >= TRAIN_SIZE:
    train_end_date = times[TRAIN_SIZE - 1].date()
else:
    train_end_date = grid_end

print("=== 时间对齐分析 ===")
print(f"网格数据范围 (UTC):")
print(f"  全部: {grid_start} ~ {grid_end}")
print(f"  训练: {grid_start} ~ {train_end_date}")
print(f"\n精确点数据范围:")
print(f"  {turbine_dates[0]} ~ {turbine_dates[-1]}")

# 计算重叠
turbine_start = datetime.strptime(str(turbine_dates[0]), "%Y-%m-%d").date()
turbine_end = datetime.strptime(str(turbine_dates[-1]), "%Y-%m-%d").date()

overlap_start = max(grid_start, turbine_start)
overlap_end = min(train_end_date, turbine_end)

if overlap_start <= overlap_end:
    overlap_days = (overlap_end - overlap_start).days + 1
    print(f"\n重叠范围: {overlap_start} ~ {overlap_end}")
    print(f"重叠天数: {overlap_days} 天")
else:
    print(f"\n警告: 无重叠!")

=== 时间对齐分析 ===
网格数据范围 (UTC):
  全部: 2020-01-01 ~ 2025-11-01
  训练: 2020-01-01 ~ 2024-12-30

精确点数据范围:
  2024-09-23 ~ 2025-08-24

重叠范围: 2024-09-23 ~ 2024-12-30
重叠天数: 99 天


In [12]:
# 统计超出范围的日期
out_of_range = []
in_range = []

for date_str in turbine_dates:
    date = datetime.strptime(str(date_str), "%Y-%m-%d").date()
    if date > train_end_date:
        out_of_range.append(date_str)
    else:
        in_range.append(date_str)

print(f"=== 日期统计 ===")
print(f"在训练集范围内: {len(in_range)} 天")
print(f"超出训练集范围: {len(out_of_range)} 天")

if out_of_range:
    print(f"\n超出范围的日期 (前5个):")
    for d in out_of_range[:5]:
        print(f"  {d}")
    if len(out_of_range) > 5:
        print(f"  ... 共 {len(out_of_range)} 天")

=== 日期统计 ===
在训练集范围内: 87 天
超出训练集范围: 179 天

超出范围的日期 (前5个):
  2024-12-31
  2025-01-01
  2025-01-02
  2025-01-03
  2025-01-04
  ... 共 179 天


## 6. 结论

In [13]:
print("="*60)
print("结论")
print("="*60)
print()
print("1. 时间格式: ERA5 数据使用 UTC 时间（非北京时间）")
print("   北京时间 = UTC + 8 小时")
print()
print("2. 对精确点数据对齐的影响:")
print("   - 如果精确点数据是北京时间，需要转换为 UTC")
print("   - 或者在对齐时考虑 8 小时偏移")
print()
print("3. 建议:")
print("   - 确认精确点数据的时区")
print("   - 如果都是 UTC，无需转换")
print("   - 如果精确点是北京时间，需统一处理")

结论

1. 时间格式: ERA5 数据使用 UTC 时间（非北京时间）
   北京时间 = UTC + 8 小时

2. 对精确点数据对齐的影响:
   - 如果精确点数据是北京时间，需要转换为 UTC
   - 或者在对齐时考虑 8 小时偏移

3. 建议:
   - 确认精确点数据的时区
   - 如果都是 UTC，无需转换
   - 如果精确点是北京时间，需统一处理
